# Chapter 3 Exercises
## Part I - Foundations of Machine Learning

These exercises reinforce beginner ideas about supervised learning, evaluation, model capacity, and fit.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = 'retina'


## Exercise 1: Regression

Fit a simple line to tiny synthetic data and compare train and test error.


In [ ]:
x = np.array([0., 1., 2., 3., 4., 5., 6., 7.])
y = np.array([0.9, 2.0, 2.8, 4.1, 5.3, 6.1, 7.4, 8.2])

x_train, x_test = x[:6], x[6:]
y_train, y_test = y[:6], y[6:]

slope, intercept = np.polyfit(x_train, y_train, deg=1)
train_pred = slope * x_train + intercept
test_pred = slope * x_test + intercept

train_mse = np.mean((y_train - train_pred) ** 2)
test_mse = np.mean((y_test - test_pred) ** 2)

print(f"slope: {slope:.3f}")
print(f"intercept: {intercept:.3f}")
print(f"train MSE: {train_mse:.3f}")
print(f"test MSE: {test_mse:.3f}")

plt.figure(figsize=(4.8, 3.4))
plt.scatter(x_train, y_train, label="train")
plt.scatter(x_test, y_test, label="test")
xx = np.linspace(x.min(), x.max(), 100)
plt.plot(xx, slope * xx + intercept, color="red", label="fit")
plt.xlabel("input")
plt.ylabel("target")
plt.legend()
plt.tight_layout()
plt.show()


The test MSE is close to the train MSE — the model generalizes rather than memorizes. This is expected when the data follows a clean linear trend with modest noise; the train-test gap tends to grow only when the model has excess capacity or the dataset is very small. A slope near 1.0 and a tight visual fit confirm that the linear form is a good match for this particular relationship.

## Exercise 2: Classification

Use a tiny flower dataset and a simple threshold rule to classify two labels.


In [ ]:
df = pd.DataFrame({
    "petal_length_cm": [1.4, 1.4, 1.5, 1.3, 1.7, 1.5, 4.5, 4.5, 4.7, 3.9, 4.7, 4.4],
    "petal_width_cm": [0.2, 0.2, 0.2, 0.2, 0.4, 0.3, 1.5, 1.5, 1.4, 1.1, 1.5, 1.3],
    "label": [
        "setosa", "setosa", "setosa", "setosa", "setosa", "setosa",
        "versicolor", "versicolor", "versicolor", "versicolor", "versicolor", "versicolor"
    ]
})

y = (df["label"] == "versicolor").astype(int).values
train_idx = np.array([0, 1, 2, 6, 7, 8, 9, 10])
test_idx = np.array([3, 4, 5, 11])

x_train = df.loc[train_idx, "petal_length_cm"].values
y_train = y[train_idx]
x_test = df.loc[test_idx, "petal_length_cm"].values
y_test = y[test_idx]

threshold = 0.5 * (x_train[y_train == 0].mean() + x_train[y_train == 1].mean())
train_pred = (x_train > threshold).astype(int)
test_pred = (x_test > threshold).astype(int)

train_acc = np.mean(train_pred == y_train)
test_acc = np.mean(test_pred == y_test)

print(f"threshold: {threshold:.2f} cm")
print(f"train accuracy: {train_acc:.2f}")
print(f"test accuracy: {test_acc:.2f}")
print("test predictions:")
print(pd.DataFrame({"petal_length_cm": x_test, "actual": y_test, "predicted": test_pred}))

plt.figure(figsize=(4.8, 3.4))
for label_name, color in [("setosa", "tab:blue"), ("versicolor", "tab:orange")]:
    subset = df[df["label"] == label_name]
    plt.scatter(subset["petal_length_cm"], subset["petal_width_cm"], color=color, label=label_name, s=50)
plt.axvline(threshold, color="black", linestyle="--", label="threshold")
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


Both train and test accuracy reach 1.00 — a single midpoint threshold on petal length perfectly separates setosa from versicolor on both splits. This tells us that petal length alone draws a completely clean boundary between these two species; no complex model is needed here. It is a useful reminder that the simplest rule that works is often the best starting point before reaching for more powerful methods.

## Exercise 3: Model Capacity

Compare simple low-capacity and higher-capacity rules on a small 1D classification task.


In [ ]:
x = np.linspace(-3, 3, 16)
y = (np.abs(x) > 1.2).astype(int)
y[[2, 12]] = 1 - y[[2, 12]]

train_idx = np.array([0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 14])
test_idx = np.array([4, 11, 13, 15])

x_train, y_train = x[train_idx], y[train_idx]
x_test, y_test = x[test_idx], y[test_idx]

def binned_majority_predict(x_train, y_train, x_query, n_bins, x_min, x_max):
    edges = np.linspace(x_min, x_max, n_bins + 1)
    default_label = int(np.round(y_train.mean()))
    bin_labels = []
    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (x_train >= edges[i]) & (x_train <= edges[i + 1])
        else:
            mask = (x_train >= edges[i]) & (x_train < edges[i + 1])
        if mask.any():
            bin_labels.append(int(np.round(y_train[mask].mean())))
        else:
            bin_labels.append(default_label)
    preds = np.empty(len(x_query), dtype=int)
    for j, value in enumerate(x_query):
        idx = np.searchsorted(edges, value, side="right") - 1
        idx = int(np.clip(idx, 0, n_bins - 1))
        preds[j] = bin_labels[idx]
    return preds

capacities = [1, 3, 8]
rows = []
grid = np.linspace(x.min(), x.max(), 300)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)

for i, n_bins in enumerate(capacities):
    train_pred = binned_majority_predict(x_train, y_train, x_train, n_bins, x.min(), x.max())
    test_pred = binned_majority_predict(x_train, y_train, x_test, n_bins, x.min(), x.max())
    grid_pred = binned_majority_predict(x_train, y_train, grid, n_bins, x.min(), x.max())
    rows.append({
        "capacity_bins": n_bins,
        "train_accuracy": np.mean(train_pred == y_train),
        "test_accuracy": np.mean(test_pred == y_test),
    })
    ax[i].scatter(x_train, y_train, label="train")
    ax[i].scatter(x_test, y_test, label="test")
    ax[i].plot(grid, grid_pred, color="red")
    ax[i].set_title(f"bins = {n_bins}")
    ax[i].set_xlabel("input")
    ax[i].set_ylim(-0.2, 1.2)

ax[0].set_ylabel("class")
ax[0].legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame(rows).round(3)


Notice the trade-off across the three plots: 1 bin predicts the same class everywhere and underfits badly, while 8 bins can memorize individual training points at the cost of test accuracy. The middle capacity (3 bins) balances the two, achieving reasonable accuracy on both splits. This is the bias-variance trade-off in compact form — more capacity reduces training error but can increase test error when the model starts fitting noise rather than signal.

## Exercise 4: Flexibility and Fit

Compare polynomial degrees to see underfitting and overfitting on a tiny regression problem.


In [ ]:
rng = np.random.default_rng(2)
x = np.linspace(-3, 3, 12)
y = 0.5 * x**2 + x + rng.normal(0, 0.8, size=len(x))

train_idx = np.array([0, 1, 2, 4, 5, 7, 8, 10])
test_idx = np.array([3, 6, 9, 11])
x_train, y_train = x[train_idx], y[train_idx]
x_test, y_test = x[test_idx], y[test_idx]

degrees = [1, 2, 5]
rows = []
xx = np.linspace(x.min(), x.max(), 300)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)

for i, degree in enumerate(degrees):
    coeffs = np.polyfit(x_train, y_train, deg=degree)
    train_pred = np.polyval(coeffs, x_train)
    test_pred = np.polyval(coeffs, x_test)
    rows.append({
        "degree": degree,
        "train_MSE": np.mean((y_train - train_pred) ** 2),
        "test_MSE": np.mean((y_test - test_pred) ** 2),
    })
    ax[i].scatter(x_train, y_train, label="train")
    ax[i].scatter(x_test, y_test, label="test")
    ax[i].plot(xx, np.polyval(coeffs, xx), color="red")
    ax[i].set_title(f"degree {degree}")
    ax[i].set_xlabel("input")

ax[0].set_ylabel("target")
ax[0].legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame(rows).round(3)


Degree 1 underfits — train and test MSE are both elevated because a straight line cannot capture the underlying quadratic curve. Degree 2 hits the sweet spot: the model has exactly the right flexibility for the true relationship, and both errors are low. Degree 5 overfits: train MSE drops toward zero as the model chases noise in the training points, while test MSE rises sharply — the classic signature of a model that has memorized rather than generalized.

## Key Takeaways

- Regression and classification should be checked with simple train and test metrics.
- A more flexible model can fit training data better, but it may not help on unseen data.
- Capacity and flexibility should be increased carefully, especially on small datasets.
- Looking at both plots and errors helps build intuition about underfitting and overfitting.


## Reflections

These exercises made the bias-variance trade-off concrete in two complementary settings: a classifier whose capacity is either too blunt or too fine-grained, and polynomial regression where degree 1 underfits and degree 5 memorizes. The consistent message is that training error alone is not a useful signal — what matters is how the gap between train and test error changes with model flexibility, and whether that gap can be closed by adding data, reducing capacity, or changing the model form. Every deep learning technique covered in Parts II and III — dropout, weight decay, batch normalization, early stopping — is ultimately a tool for managing this same gap. Understanding it here with the simplest possible models is what makes those techniques legible rather than magical when you encounter them later.